In [1]:
from scripts.conf_file_finding import try_find_conf_file
try_find_conf_file()

Local configuration file found !!, no need to run the configuration (unless configuration has changed)


In [2]:
import os
from scipy.io import savemat

import datajoint as dj
dj.conn()


/mnt/cup/braininit/Shared/repos/TestU19PipelinePython2/U19-pipeline-python/.venv/lib/python3.13/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-05 12:02:49,980][INFO]: DataJoint 0.14.6 connected to alvaros@datajoint00.pni.princeton.edu:3306


DataJoint connection (connected) alvaros@datajoint00.pni.princeton.edu:3306

In [5]:
ephys_element =dj.create_virtual_module('u19_pipeline_ephys_element','u19_pipeline_ephys_element')
imaging_element =dj.create_virtual_module('u19_pipeline_imaging_element','u19_pipeline_imaging_element')

modalities             = ['electrophysiology', 'imaging']

params_tables          = [ephys_element.ClusteringParamSet, imaging_element.ProcessingParamSet]
preparams_steps_tables = [(ephys_element.PreClusterParamSteps * ephys_element.PreClusterParamSteps.Step * ephys_element.PreClusterParamSet)]
preparams_tables       = [ephys_element.PreClusterParamSet]


In [6]:
params_dict_list = []
for table in params_tables:
    params_dict_list.append(table.fetch(as_dict=True))


#Append all params in the same dictionary
params_dict_dict = {}
num_params = 0
for idx, param_modality_list in enumerate(params_dict_list):
    for dicto in param_modality_list:
        dicto['recording_modality'] = modalities[idx]
        dicto['param_set_hash'] = str(dicto['param_set_hash'])
        if 'clustering_method' in dicto:
            dicto['processing_method'] = dicto.pop('clustering_method')
    
        params_dict_dict['param_'+str(num_params)] = dicto
        num_params +=1

#################################################Fetch all preparamsStepList from all modalities
preparams_steps = []
for table in preparams_steps_tables:
    preparams_steps.append(table.fetch(as_dict=True))

#Append all preparams in the same dictionary
preparams_steps_dict_dict = {}
num_preparams_steps = 0
for idx, preparam_modality_list in enumerate(preparams_steps):
    for dicto in preparam_modality_list:
        dicto['param_set_hash'] = str(dicto['param_set_hash'])
        dicto['recording_modality'] = modalities[idx]
        if 'precluster_param_steps_id' in dicto:
            dicto['preprocess_param_steps_id'] = dicto.pop('precluster_param_steps_id')
        if 'precluster_method' in dicto:
            dicto['preprocess_method'] = dicto.pop('precluster_method')
        if 'precluster_param_steps_name' in dicto:
            dicto['preprocess_param_steps_name'] = dicto.pop('precluster_param_steps_name')
        if 'precluster_param_steps_desc' in dicto:
            dicto['preprocess_param_steps_desc'] = dicto.pop('precluster_param_steps_desc')

        preparams_steps_dict_dict['param_'+str(num_preparams_steps)] = dicto
        num_preparams_steps +=1

#################################################Fetch all preparams from all modalities
preparams_dict_list = []
for table in preparams_tables:
    preparams_dict_list.append(table.fetch(as_dict=True))


#Append all preparams in the same dictionary
preparams_dict_dict = {}
num_preparams = 0
for idx, preparam_modality_list in enumerate(preparams_dict_list):
    for dicto in preparam_modality_list:
        dicto['recording_modality'] = modalities[idx]
        dicto['param_set_hash'] = str(dicto['param_set_hash'])
        if 'precluster_method' in dicto:
            dicto['preprocess_method'] = dicto.pop('precluster_method')
    
        preparams_dict_dict['param_'+str(num_preparams)] = dicto
        num_preparams +=1


In [7]:
def replace_none_inplace(data, replacement=[]):
    """Recursively replaces None values in a dictionary in-place."""
    if isinstance(data, dict):
        for key, value in data.items():
            if value is None:
                data[key] = replacement
            elif isinstance(data, dict):
                replace_none_inplace(value, replacement)
    elif isinstance(data, list):
        for index, item in enumerate(data):
            if item is None:
                data[index] = replacement
            elif isinstance(item, (dict, list)):
                replace_none_inplace(item, replacement)

In [8]:
replace_none_inplace(params_dict_dict)

In [9]:
params_dict_dict

{'param_0': {'paramset_idx': 0,
  'paramset_desc': 'general-user_2022-06-01_Spike sorting using Kilosort2 Old',
  'param_set_hash': 'b9e07e55-95ea-3463-a740-90291de41da6',
  'params': {'fs': 30000,
   'fshigh': 150,
   'minfr_goodchannels': 0.1,
   'Th': [10, 4],
   'lam': 10,
   'AUCsplit': 0.9,
   'minFR': 0.02,
   'momentum': [20, 400],
   'sigmaMask': 30,
   'ThPre': 8,
   'CAR': 1,
   'spkTh': -6,
   'reorder': 1,
   'nskip': 25,
   'GPU': 1,
   'Nfilt': 1024,
   'nfilt_factor': 4,
   'ntbuff': 64,
   'NT': 32832,
   'whiteningRange': 32,
   'nSkipCov': 25,
   'scaleproc': 200,
   'nPCs': 3,
   'useRAM': 0,
   'trange': [0, 1000000000],
   'NchanTOT': 385},
  'recording_modality': 'electrophysiology',
  'processing_method': 'kilosort2'},
 'param_1': {'paramset_idx': 1,
  'paramset_desc': 'alvaros_2022-06-01_Spike sorting using Kilosort2',
  'param_set_hash': '697d24ac-2d46-a6e8-6f6b-2d1afaea000d',
  'params': {'fs': 30000,
   'fshigh': 150,
   'minfr_goodchannels': 0.1,
   'Th': [

In [10]:
dj.conn().close()


savemat('params.mat', params_dict_dict)
savemat('preparams.mat', preparams_dict_dict)
savemat('preparams_list.mat', preparams_steps_dict_dict)

/mnt/cup/braininit/Shared/repos/TestU19PipelinePython2/U19-pipeline-python/.venv/lib/python3.13/site-packages/scipy/io/matlab/_mio5.py:659: MatWriteWarning: Starting field name with a underscore or a digit (1Preg) is ignored
  narr = to_writeable(arr)
